# Heliyon Revision — Experiment Runner

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

1. Run the upload cell and select `bundle.zip`.
2. Runtime → Run all. Total ≈ 2–4 h on a T4.
3. The last cell downloads `outputs_revision.zip` — save it and hand it back for local integration.

Stages are resume-safe: if the session disconnects, re-upload the bundle, re-run all cells, and completed stages (detected from the partial zip you re-upload as `outputs_revision_partial.zip`, optional) are skipped.

In [ ]:
from google.colab import files
import os
print('Upload bundle.zip')
up = files.upload()
fname = list(up.keys())[0]          # accept any saved name, e.g. "bundle (2).zip"
os.replace(fname, 'bundle_upload.zip')
!rm -rf bundle && unzip -qo bundle_upload.zip -d bundle && ls bundle

In [ ]:
# Resume: to continue an interrupted/previous session, upload your saved
# outputs_revision.zip when prompted (skip with Cancel if starting fresh).
# The zip's internal paths already carry bundle/outputs_revision/, so it
# unzips at /content and restores completed-stage markers + checkpoints.
from google.colab import files
import os
print('OPTIONAL: upload outputs_revision.zip from a previous run (or Cancel)')
up = files.upload()
for fname in up:
    os.replace(fname, 'previous_outputs.zip')
    !unzip -qo previous_outputs.zip -d /content
    print('previous outputs restored:')
    !ls bundle/outputs_revision 2>/dev/null | head -20

In [ ]:
%pip install -q shap xgboost lightgbm pandapower
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Smoke test first (~10 min)
Validates every stage on a tiny dataset before committing hours to the full run.
Expect `MISMATCH` flags in stage0 **here** — the tiny dataset cannot reproduce the
published numbers; what matters is that every stage prints `[DONE]` without errors.

In [ ]:
%cd /content/bundle
!python run_all.py --smoke --stages stage0,a1,a3,a2,a4,a5,a9,a8

## Full run (~2–4 h on T4)
Only proceed if the smoke test above completed every stage. In **this** stage0,
all flags should be `OK` (or `WARN` within 2 pp for the retrained CNN) — a
`MISMATCH` means the published results did not reproduce and must be
investigated before anything else is trusted.

In [ ]:
%cd /content/bundle
!python run_all.py --stages stage0,a1,a3,a2,a4,a5,a9

In [ ]:
# Dataset export for Zenodo (~1 GB compressed). Set RUN_DATASET_EXPORT = True to include.
RUN_DATASET_EXPORT = True
if RUN_DATASET_EXPORT:
    !python run_all.py --stages a8

In [ ]:
%cd /content
!zip -qr outputs_revision.zip bundle/outputs_revision -x '*dataset_export*'
from google.colab import files
files.download('outputs_revision.zip')
import os
if os.path.exists('bundle/outputs_revision/dataset_export/power_grid_fault_benchmark_50k.npz'):
    !zip -qjr dataset_export.zip bundle/outputs_revision/dataset_export
    files.download('dataset_export.zip')